Pruebas de validación de pepBERT 
1. Balanceo de datos
2. trabajo de pruebas del modelo preentrenado

In [25]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv, find_dotenv

# ¡LA MAGIA ESTÁ AQUÍ! 
# find_dotenv() escanea hacia atrás (hacia la carpeta accelerated-drug-design) buscando el .env
load_dotenv(find_dotenv(), override=True) 

# Buscar la variable exacta
ruta_dataset = os.getenv('ruta_dataset2')

# --- FILTRO DE SEGURIDAD ---
if ruta_dataset is None:
    print("❌ ERROR: Aún no encuentro 'ruta_dataset2'.")
    print("Revisa que el archivo .env esté guardado y que la variable se llame exactamente igual.")
elif not os.path.exists(ruta_dataset):
    print(f"❌ ERROR: Encontré la ruta en el .env, pero el CSV no está físicamente en este lugar: {ruta_dataset}")
else:
    # Si pasa las pruebas, cargamos el dataset
    print("✅ ¡Ruta .env y archivo encontrados! Cargando el Dataset...", '----'*10)
    df = pd.read_csv(ruta_dataset)
    print(f"🎉 Dataset cargado exitosamente. Tamaño: {df.shape}")
    display(df.head(2)) # Muestra las primeras 2 filas para confirmar

✅ ¡Ruta .env y archivo encontrados! Cargando el Dataset... ----------------------------------------
🎉 Dataset cargado exitosamente. Tamaño: (28800, 8)


,mpnn,plddt,ptm,i_ptm,pae,i_pae,rmsd,seq
0,1.610,0.759,0.716,0.687,6.965,7.368,5.898,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
1,1.593,0.589,0.664,0.493,10.559,11.163,1.549,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...


In [26]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv

# 1. Le decimos a Python que el archivo está una carpeta atrás (..) y se llama rutas.env
ruta_archivo_env = os.path.join(os.getcwd(), "..", "rutas.env")

# 2. Forzamos a dotenv a leer ESE archivo específico
load_dotenv(dotenv_path=ruta_archivo_env, override=True)

# 3. Ahora sí, pedimos nuestra variable secreta
ruta_dataset = os.getenv('ruta_dataset2')

# --- FILTRO DE SEGURIDAD ---
if ruta_dataset is None:
    print(f"❌ ERROR: No pude encontrar la variable 'ruta_dataset2' dentro de tu archivo rutas.env")
    print(f"Asegúrate de que el archivo exista en: {ruta_archivo_env}")
elif not os.path.exists(ruta_dataset):
    print(f"❌ ERROR: La variable se cargó en secreto, pero el CSV no está físicamente en esa ruta.")
else:
    print("🔒 Leyendo configuración segura desde rutas.env...")
    print("✅ ¡Ruta secreta cargada con éxito!", '----'*10)
    df = pd.read_csv(ruta_dataset)
    print(f"🎉 Dataset cargado. Tamaño: {df.shape}")
    display(df.head(5))

🔒 Leyendo configuración segura desde rutas.env...
✅ ¡Ruta secreta cargada con éxito! ----------------------------------------


🎉 Dataset cargado. Tamaño: (28800, 8)


,mpnn,plddt,ptm,i_ptm,pae,i_pae,rmsd,seq
0,1.610,0.759,0.716,0.687,6.965,7.368,5.898,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
1,1.593,0.589,0.664,0.493,10.559,11.163,1.549,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
2,1.599,0.787,0.728,0.721,6.648,7.012,5.711,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
3,1.578,0.759,0.716,0.687,6.965,7.368,5.898,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...
4,1.656,0.762,0.706,0.696,7.129,7.536,4.984,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...


In [27]:
df.shape

(28800, 8)

In [28]:
# --- CELDA 2: BALANCEO DE DATOS Y VISUALIZACIÓN ---
def balancear_regresion_continua(df_input, target_col='i_ptm', umbral_asimetria=1.0):
    df_out = df_input.copy()
    sesgo = df_out[target_col].skew()
    
    print(f"📊 Analizando distribución de '{target_col}'...")
    print(f"   Coeficiente de Asimetría (Skewness): {sesgo:.3f}")
    
    if abs(sesgo) <= umbral_asimetria:
        print("   ✅ Distribución equilibrada. Asignando peso neutral.")
        df_out['sample_weight'] = 1.0
        return df_out

    print("   ⚠️ Desbalanceo detectado. Calculando percentiles exactos...\n")
    
    # 1. Calculamos dónde están las "fronteras" de los datos
    p90 = df_out[target_col].quantile(0.90)
    p95 = df_out[target_col].quantile(0.95)
    p99 = df_out[target_col].quantile(0.99)
    
    print(f"   🎯 Frontera Top 10% (p90): i_ptm >= {p90:.4f}")
    print(f"   🎯 Frontera Top  5% (p95): i_ptm >= {p95:.4f}")
    print(f"   🎯 Frontera Top  1% (p99): i_ptm >= {p99:.4f}\n")
    
    # 2. Aplicamos las reglas
    condiciones = [
        df_out[target_col] >= p99,
        df_out[target_col] >= p95,
        df_out[target_col] >= p90
    ]
    pesos = [25.0, 10.0, 3.0]
    
    # Asignamos los pesos (el resto recibe 1.0)
    df_out['sample_weight'] = np.select(condiciones, pesos, default=1.0)
    return df_out

# Ejecutamos la función
df_balanceado = balancear_regresion_continua(df, target_col='i_ptm')

# 3. CREAMOS LA TABLA DE TRANSPARENCIA PARA TI
print("="*60)
print("🔍 TABLA DE TRANSPARENCIA: ¿CÓMO SE ASIGNARON LOS PESOS?")
print("="*60)

# Agrupamos los datos por el peso que recibieron
resumen = df_balanceado.groupby('sample_weight').agg(
    Secuencias_Reales=('i_ptm', 'count'),          # Cuántas filas cayeron aquí
    i_ptm_Minimo=('i_ptm', 'min'),                 # El valor más bajo de este grupo
    i_ptm_Maximo=('i_ptm', 'max'),                 # El valor más alto de este grupo
    Atencion_Efectiva=('sample_weight', 'sum')     # Filas Reales * Peso (El poder de aprendizaje)
).reset_index()

# Le damos formato bonito para imprimirlo
print(resumen.to_string(index=False))
print("="*60)

# 4. Mostramos una muestra real de los datos élite
print("\n Muestra real de 3 secuencias ÉLITE (Peso 25):")
display(df_balanceado[df_balanceado['sample_weight'] == 25.0][['seq', 'i_ptm', 'sample_weight']].head(3))

print("\n Muestra real de 3 secuencias BASURA (Peso 1):")
display(df_balanceado[df_balanceado['sample_weight'] == 1.0][['seq', 'i_ptm', 'sample_weight']].head(3))

📊 Analizando distribución de 'i_ptm'...
   Coeficiente de Asimetría (Skewness): 0.122
   ✅ Distribución equilibrada. Asignando peso neutral.
🔍 TABLA DE TRANSPARENCIA: ¿CÓMO SE ASIGNARON LOS PESOS?
 sample_weight  Secuencias_Reales  i_ptm_Minimo  i_ptm_Maximo  Atencion_Efectiva
           1.0              28800         0.067         0.785            28800.0

 Muestra real de 3 secuencias ÉLITE (Peso 25):


,seq,i_ptm,sample_weight



 Muestra real de 3 secuencias BASURA (Peso 1):


,seq,i_ptm,sample_weight
0,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...,0.687,1.0
1,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...,0.493,1.0
2,GYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTN...,0.721,1.0


In [29]:
# --- CELDA 3: CARGAR MODELO PÚBLICO (PROT_BERT) ---
import torch
# CAMBIO CLAVE: Usamos las clases exactas en lugar de las "Auto"
from transformers import BertTokenizer, BertModel

print("🔍 Verificando hardware del servidor...")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    nombre_gpu = torch.cuda.get_device_name(0)
    device = torch.device('cuda')
    print(f"✅ ¡GPU Detectada! Tienes {num_gpus} tarjeta(s) disponible(s).")
    print(f"🚀 Usando la tarjeta: {nombre_gpu}")
else:
    device = torch.device('cpu')
    print("⚠️ No se detectó GPU activa de NVIDIA.")

print("\nCargando modelo público ProtBERT... esto tomará un momento...")

model_name = "Rostlab/prot_bert" 

# Al usar BertTokenizer, evitamos el error de sentencepiece por completo
tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False)
model = BertModel.from_pretrained(model_name)

model = model.to(device)
print(f"✅ ¡Modelo cargado exitosamente en el {device.type.upper()}!")

🔍 Verificando hardware del servidor...
✅ ¡GPU Detectada! Tienes 1 tarjeta(s) disponible(s).
🚀 Usando la tarjeta: NVIDIA RTX A4000

Cargando modelo público ProtBERT... esto tomará un momento...


Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ¡Modelo cargado exitosamente en el CUDA!


In [30]:
# --- CELDA 4: EXTRAER EMBEDDINGS (PRUEBA CON 5 SECUENCIAS) ---

# 1. Tomamos solo las primeras 5 secuencias de tu dataset balanceado
df_prueba = df_balanceado.head(5).copy()

# 2. Función para limpiar y espaciar la secuencia para ProtBERT
def preparar_secuencia_para_bert(secuencia_cruda):
    # Separamos las dos cadenas usando la '/'
    cadenas = secuencia_cruda.split('/')
    
    # Le ponemos un espacio entre cada letra a cada cadena: 'ABC' -> 'A B C'
    cadenas_espaciadas = [" ".join(list(cad)) for cad in cadenas]
    
    # Las volvemos a unir, pero ahora usando el token [SEP] en medio
    secuencia_final = " [SEP] ".join(cadenas_espaciadas)
    return secuencia_final

# Aplicamos la limpieza a nuestras 5 secuencias
df_prueba['seq_bert'] = df_prueba['seq'].apply(preparar_secuencia_para_bert)

print("👀 Ejemplo de cómo BERT ve tu secuencia ahora:")
print(f"Original: {df_prueba['seq'].iloc[0][:15]}...")
print(f"Para BERT: {df_prueba['seq_bert'].iloc[0][:25]}...\n")

# 3. Función para traducir a números
def obtener_embedding(secuencia_formateada):
    # Convertimos el texto al formato matemático de PyTorch
    inputs = tokenizer(secuencia_formateada, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Pasamos la secuencia por la IA sin entrenarla
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extraemos el vector [CLS] (posición 0), que es el resumen de TODA la proteína
    embedding = outputs.last_hidden_state[0, 0, :].cpu().numpy()
    return embedding

print("⚙️ Traduviendo 5 secuencias a vectores matemáticos...")
embeddings = [obtener_embedding(seq) for seq in df_prueba['seq_bert']]

print("🎉 ¡Éxito total!")
print(f"📏 Tamaño de un solo embedding: {embeddings[0].shape}")
print(f"🔢 Muestra de los primeros 4 números extraídos:\n{embeddings[0][:4]}")

👀 Ejemplo de cómo BERT ve tu secuencia ahora:
Original: GYPKAEVIWTSSDHQ...
Para BERT: G Y P K A E V I W T S S D...

⚙️ Traduviendo 5 secuencias a vectores matemáticos...
🎉 ¡Éxito total!
📏 Tamaño de un solo embedding: (1024,)
🔢 Muestra de los primeros 4 números extraídos:
[-0.02255799  0.10531124  0.01840047 -0.1344108 ]


In [31]:
# --- CELDA 5 OPTIMIZADA: EXTRACCIÓN MASIVA Y GUARDADO DIRECTO ---
import os
import numpy as np
from tqdm import tqdm
import torch
from dotenv import load_dotenv

# 1. Cargar ruta de destino primero
ruta_archivo_env = os.path.join(os.getcwd(), "..", "rutas.env")
load_dotenv(dotenv_path=ruta_archivo_env, override=True)
ruta_resultados = os.getenv('ruta_resultados')

if ruta_resultados is None:
    print(f"❌ ERROR: No encontré 'ruta_resultados' en el archivo: {ruta_archivo_env}")
    # Detenemos la ejecución si no hay ruta
    raise ValueError("Falta la ruta de resultados en rutas.env") 

# Crear la carpeta de resultados si no existe
os.makedirs(ruta_resultados, exist_ok=True)

# 2. Preparar secuencias
print("🧹 Limpiando todas las secuencias para BERT...")
df_balanceado['seq_bert'] = df_balanceado['seq'].apply(preparar_secuencia_para_bert)
secuencias_totales = df_balanceado['seq_bert'].tolist()

batch_size = 32
todos_los_embeddings = []

print(f"🚀 Iniciando extracción masiva para {len(secuencias_totales)} secuencias...")
model.eval()

# 3. Extracción
for i in tqdm(range(0, len(secuencias_totales), batch_size), desc="Extrayendo Embeddings"):
    batch_seqs = secuencias_totales[i : i + batch_size]
    inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    todos_los_embeddings.extend(batch_embeddings)

matriz_embeddings = np.array(todos_los_embeddings)
print(f"\n✅ Extracción terminada. Dimensión: {matriz_embeddings.shape}")

# 4. GUARDADO DIRECTO EN LA RUTA FINAL
# Construimos la ruta completa combinando la carpeta de resultados y el nombre del archivo
ruta_final_npy = os.path.join(ruta_resultados, 'embeddings_protbert_28k.npy')

np.save(ruta_final_npy, matriz_embeddings)
print(f"💾 Embeddings guardados DIRECTAMENTE de forma segura en:\n   {ruta_final_npy}")

🧹 Limpiando todas las secuencias para BERT...


🚀 Iniciando extracción masiva para 28800 secuencias...


Extrayendo Embeddings: 100%|██████████| 900/900 [05:57<00:00,  2.51it/s]



✅ Extracción terminada. Dimensión: (28800, 1024)
💾 Embeddings guardados DIRECTAMENTE de forma segura en:
   /home/a01796407/proyectos_codigo/accelerated-drug-design/data/resultados/embeddings/embeddings_protbert_28k.npy


In [ ]:
# --- CELDA 6: ENTRENANDO EL PREDICTOR (RED NEURONAL MLP) ---
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

print("🧠 Preparando datos para la Red Neuronal...")

# 1. Cargar la ruta de forma segura
ruta_archivo_env = os.path.join(os.getcwd(), "..", "rutas.env")
load_dotenv(dotenv_path=ruta_archivo_env, override=True)
ruta_resultados = os.getenv('ruta_resultados')

# 2. Cargar los números de BERT
ruta_final_npy = os.path.join(ruta_resultados, "embeddings_protbert_28k.npy")
X_embeddings = np.load(ruta_final_npy)

# 3. Agregar la variable de energía ('mpnn')
# Convertimos la columna a matriz y la pegamos junto a los 1024 números de BERT
X_mpnn = df_balanceado['mpnn'].values.reshape(-1, 1)
X_final = np.hstack((X_embeddings, X_mpnn)) # Ahora X tiene 1025 columnas

# 4. Definir lo que queremos predecir (Y) y los pesos para que el modelo no sea flojo
y = df_balanceado['i_ptm'].values
pesos = df_balanceado['sample_weight'].values

# 5. Dividir los datos: 80% para estudiar, 20% para el examen final
# Le pasamos también los pesos para que se dividan igual
X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_final, y, pesos, test_size=0.20, random_state=42
)

# 6. Escalar los datos (Paso OBLIGATORIO para Redes Neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Crear el "Cerebro" (MLP)
print(f"🚀 Entrenando modelo con {X_train.shape[0]} secuencias de entrenamiento...")
print("⏳ Va empzar puede tardar..")

mlp_model = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128), # 3 capas de neuronas (profundas)
    activation='relu',                  # Función de activación estándar
    solver='adam',                      # Optimizador ultrarrápido
    max_iter=300,                       # Límite de vueltas de estudio
    random_state=42,
    early_stopping=True                 # ¡Truco! Si ya no aprende más, se detiene antes para ahorrar tiempo
)

# 8. ¡EL ENTRENAMIENTO! (Aquí inyectamos tus pesos de la élite)
# Ojo: Scikit-learn en su MLP a veces no acepta sample_weights directamente en todos los entornos. 
# Si tu versión lo acepta, lo usamos.
try:
    mlp_model.fit(X_train_scaled, y_train, sample_weight=pesos_train)
except TypeError:
    print("⚠️ Tu versión de Scikit-Learn no soporta pesos en MLP directamente. Entrenando sin pesos...")
    mlp_model.fit(X_train_scaled, y_train)

# 9. El Examen Final (Predicciones sobre los datos que nunca ha visto)
print("🎯 Evaluando el modelo...")
predicciones = mlp_model.predict(X_test_scaled)

# 10. Las calificaciones
mae = mean_absolute_error(y_test, predicciones)
spearman_corr, _ = spearmanr(y_test, predicciones)

print("\n" + "="*50)
print("🏆 RESULTADOS DEL PREDICTOR (i_PTM)")
print("="*50)
print(f"📉 Error Absoluto Medio (MAE): {mae:.4f}")
print(f"📈 Correlación de Spearman:    {spearman_corr:.4f}")
print("="*50)

# 11. Mostramos cómo se vería el Ranking de las mejores proteínas
resultados_df = pd.DataFrame({
    'i_ptm_Real': y_test,
    'i_ptm_Predicho': predicciones
})
# Ordenamos de mayor a menor según la predicción de nuestra IA
top_200_simulado = resultados_df.sort_values(by='i_ptm_Predicho', ascending=False).head(10)

print("\n👑 Muestra del Top 10 Simulado (El que mandarías a AlphaFold):")
display(top_200_simulado)

🧠 Preparando datos para la Red Neuronal...
🚀 Entrenando modelo con 23040 secuencias de entrenamiento...
⏳ Va empzar puede tardar..
